### Libraries



In [1]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub \
  evaluate

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import hf_hub_download, login
import pandas as pd
from sklearn.metrics import classification_report
from collections import Counter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.6 MB/s eta 0:00:00


### Login to huggingface

In [2]:
login()

### Testing

In [ ]:
# Use GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Load tokenizer and model with LoRA
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_repo_id  = "eduhuemar001/tinyllama-german-checkpoints-sentiment"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id)
lora_model = PeftModel.from_pretrained(base_model, lora_repo_id)
lora_model = lora_model.to(device)
lora_model.eval()

cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(base_model_id)
model = model.to(device)
model.eval()

cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb): 

In [4]:
csv_path = hf_hub_download(
    repo_id="eduhuemar001/sentiment-GermEval2017",
    filename="germeval2017_cleaned.csv",
    repo_type="dataset"
)

# Load dataset
df = pd.read_csv(csv_path)
df = df[["review_text", "sentiment"]]
df = df.dropna(subset=["review_text", "sentiment"])
df["review_text"] = df["review_text"].astype(str).str.strip()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
df = df[df["sentiment"].isin(["positive", "neutral", "negative"])]

# Sample equally from all 3 classes
samples_per_class = 66
dfs = []

for label in ["positive", "neutral", "negative"]:
    class_df = df[df["sentiment"] == label]
    if len(class_df) < samples_per_class:
        raise ValueError(f"Not enough samples for class '{label}'")
    dfs.append(class_df.sample(n=samples_per_class, random_state=42))

df_balanced = pd.concat(dfs).sample(frac=1, random_state=42).reset_index(drop=True)

# Prompt template
instruction = (
    "### Instruction:\n"
    "Klassifiziere die Stimmung der folgenden Bewertung als 'positive', 'neutral' oder 'negative'.\n\n"
    "### Bewertung:\n"
)
answer_prefix = "\n\n### Antwort:\n"

# Run model inference
model = model.to(device)
model.eval()

true_labels = []
pred_labels = []

for i, row in df_balanced.iterrows():
    text = row["review_text"]
    true_label = row["sentiment"]

    prompt = instruction + text + answer_prefix
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    #output = model.generate(**inputs, max_new_tokens=2)
    output = lora_model.generate(**inputs, max_new_tokens=2)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract prediction
    if "### Antwort:" in decoded:
        answer = decoded.split("### Antwort:")[-1].strip().lower()
        answer = answer.split()[0] if answer.split() else ""
    else:
        answer = decoded.strip().lower()

    # Record results
    true_labels.append(true_label)
    pred_labels.append(answer if answer in ["positive", "neutral", "negative"] else "neutral")

    print(f"\n[{i+1}] Bewertung: {text}")
    print(f"    Wahre Stimmung: {true_label}")
    print(f"    Modellantwort: {answer}")

# Classification report
print("\nKlassifikationsbericht")
print(classification_report(true_labels, pred_labels, digits=3))

germeval2017_cleaned.csv:   0%|          | 0.00/10.7M [00:00<?, ?B/s]


[1] Bewertung: In Berlin mit der S-Bahn zum Event sechs S-Bahn-Linien (S41, S42, S46, S5, S7, S75) ermöglichen eine stressfreie Anreise zu den IFA-Bahnhöfen Messe Süd, Messe Nord/ICC und Westkreuz. Vom Berliner
    Wahre Stimmung: positive
    Modellantwort: bewert

[2] Bewertung: Wohnung, Dachgeschosswohnung in 16341 Panketal zum Kauf | Anlageobjekt vor den Toren von Berlin - vermietete Die angebotene Zweiraumwohnung ist vermietet und befindet sich in einen 1995 gebauten Wohnanlage. Die Wohnung befindet sich in einen sehr gepflegten Zustand und verfügt über ein helles Wohnzimmer mit integrierter Küchenzeile, Schlafzimmer sowie Badez
    Wahre Stimmung: neutral
    Modellantwort: positive

[3] Bewertung: Re: Deutsche Bahn Konzern "Eigentlich hat die schaffnerjn alles richtig gemacht.... Ihre Tochter hatte kejne ,,gültige"" Fahrkarte und hat eben die Fahrpreis nach erhebung bekommen... Was oder wie alles genau ablief wissen nur die schaffnerjn und ihre Tochter .. Und zum Thema ans Tele

Token indices sequence length is longer than the specified maximum sequence length for this model (3552 > 2048). Running this sequence through the model will result in indexing errors
This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.



[110] Bewertung: RT @TAOnline: Die Bahn feiert heute Einweihung der neuen Hochgeschwindigkeits-Trasse Erfurt-Leipzig/Halle https://t.co/mf33QUZCAj
    Wahre Stimmung: positive
    Modellantwort: positive

[111] Bewertung: UNSERE DEUTSCHEN TRAUMATISIERTEN FLÜCHTLINGE V... UNSERE DEUTSCHEN TRAUMATISIERTEN FLÜCHTLINGE VON MIR ÜBERARBEITETER DIALOG AUS LUPO CATTIVO: Markiko sagte Ich bin mir nicht ganz sicher, aber meine Eltern haben ihren Mund ja auch nie aufgemacht und sie waren als junge Menschen dabei. Mein Vater stand mit 17 vor Hitler bevor er nach Russland geschickt wurde und meine Mutter war wohl (ich weiss es nicht ganz genau in der Kinderlandverschickung), das war bevor die beiden sich kennenlernten. Leider weiss man nix Genaues, ja und leider ist keiner mehr da den man fragen könnte. Beide schon längst nicht mehr auf dieser Welt (aber hoffentlich da wo sie sich nicht mehr so schinden müssen). Jetzt ist mir gerade eingefallen, wo wir schon mal beim Thema sind: Mein Vater ist Apr

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
